# FIFA World Cup — Analytics Feature Engineering

**Architecture:**
```
matches_clean
    ↓
team_matches          (1 row = 1 team × 1 match)
    ↓
fact_team_historical_stats   (1 row = 1 team, all-time)
fact_team_year_stage_stats   (1 row = 1 team × year × stage)
fact_modern_team_stats       (1 row = 1 team, 2002+)
fact_team_ml_features        (1 row = 1 team, clustering + ML)
```

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

## 2. Load Raw Data

In [ ]:
try:
    DATA_DIR = Path(__file__).resolve().parent.parent / 'data'
except NameError:
    DATA_DIR = Path('../data')

MIN_MODERN_MATCHES = 10  # ≈ times que chegaram ao mata-mata pelo menos uma vez desde 2002

matches = pd.read_csv(DATA_DIR / 'processed' / 'matches_clean.csv')
worldcup = pd.read_csv(DATA_DIR / 'raw' / 'world_cup.csv')

matches['Date'] = pd.to_datetime(matches['Date'], errors='coerce')

# Intervalos resultantes do pd.cut:
#   [1930-1970] Classic Era  — 1970 cai aqui porque include_lowest=True aplica-se ao primeiro bin
#   (1970-2002] Modern Era   — 2002 cai aqui
#   (2002-2026] Contemporary Era
era_bins = [1930, 1970, 2002, 2026]
era_labels = ['Classic Era', 'Modern Era', 'Contemporary Era']
matches['Era'] = pd.cut(matches['Year'], bins=era_bins, labels=era_labels, include_lowest=True)

# Knockout stages definition — reused throughout
KNOCKOUT_STAGES = ['Oitavas de Final', 'Quartas de Final', 'Semi-finais', 'Final']

# TEAM_NAME_MAP: simplificação analítica que une estados sucessores aos predecessores.
# West Germany → Germany: amplamente aceito (continuidade institucional da DFB).
# Soviet Union → Russia, Yugoslavia → Serbia, Czechoslovakia → Czech Republic:
#   continuidade institucional é debatível; alternativas são manter separados ou
#   usar nomes compostos (ex: "Russia/USSR"). Aqui optou-se pela agregação simples.
TEAM_NAME_MAP = {
    'West Germany': 'Germany',
    'Soviet Union': 'Russia',
    'Yugoslavia': 'Serbia',
    'Czechoslovakia': 'Czech Republic',
}

print(f'Matches: {len(matches)} | World Cups: {len(worldcup)}')
matches.head(3)

## 3. Intermediate Layer — `team_matches`

Granularity: **1 row = 1 team × 1 match**

This normalizes home/away into a single perspective, eliminating all downstream
home-vs-away duplication.

In [ ]:
home = matches[[
    'home_team', 'away_team', 'home_score', 'away_score',
    'h_yellow_card_long_count', 'h_red_card_count', 'h_yellow_red_card_count',
    'Stage', 'Year', 'Era'
]].copy()
home.columns = [
    'Team', 'Opponent', 'Goals_For', 'Goals_Against',
    'Yellow_Cards', 'Red_Cards_Direct', 'Red_Cards_YR',
    'Stage', 'Year', 'Era'
]

away = matches[[
    'away_team', 'home_team', 'away_score', 'home_score',
    'a_yellow_card_long_count', 'a_red_card_count', 'a_yellow_red_card_count',
    'Stage', 'Year', 'Era'
]].copy()
away.columns = [
    'Team', 'Opponent', 'Goals_For', 'Goals_Against',
    'Yellow_Cards', 'Red_Cards_Direct', 'Red_Cards_YR',
    'Stage', 'Year', 'Era'
]

team_matches = pd.concat([home, away], ignore_index=True)

# Preserve original names before consolidation for downstream drill-down
team_matches['Team_Original'] = team_matches['Team']

team_matches['Team'] = team_matches['Team'].replace(TEAM_NAME_MAP)
team_matches['Opponent'] = team_matches['Opponent'].replace(TEAM_NAME_MAP)
team_matches['Red_Cards'] = team_matches['Red_Cards_Direct'] + team_matches['Red_Cards_YR']
team_matches = team_matches.drop(columns=['Red_Cards_Direct', 'Red_Cards_YR'])

team_matches['Result'] = np.where(
    team_matches['Goals_For'] > team_matches['Goals_Against'], 'W',
    np.where(team_matches['Goals_For'] == team_matches['Goals_Against'], 'D', 'L')
)
team_matches['Is_Knockout'] = team_matches['Stage'].isin(KNOCKOUT_STAGES).astype(int)
team_matches['Knockout_Win'] = (
    (team_matches['Result'] == 'W') & (team_matches['Is_Knockout'] == 1)
).astype(int)

# Indicator columns for vectorized groupby (avoids slow lambda, ~2x faster)
team_matches['Is_Win'] = (team_matches['Result'] == 'W').astype(int)
team_matches['Is_Draw'] = (team_matches['Result'] == 'D').astype(int)
team_matches['Is_Loss'] = (team_matches['Result'] == 'L').astype(int)

print(f'team_matches shape: {team_matches.shape}  (expected ~{len(matches)*2} rows)')
team_matches.head(5)

## 4. `fact_team_historical_stats`

Granularity: **1 row = 1 team, all-time**

In [ ]:
fact_team_historical_stats = (
    team_matches.groupby('Team').agg(
        matches_played=('Is_Win', 'count'),
        wins=('Is_Win', 'sum'),
        draws=('Is_Draw', 'sum'),
        losses=('Is_Loss', 'sum'),
        goals_scored=('Goals_For', 'sum'),
        goals_conceded=('Goals_Against', 'sum'),
        total_yellow_cards=('Yellow_Cards', 'sum'),
        total_red_cards=('Red_Cards', 'sum'),
        knockout_matches=('Is_Knockout', 'sum'),
        knockout_wins=('Knockout_Win', 'sum'),
    )
    .reset_index()
)

fact_team_historical_stats['win_rate'] = (
    fact_team_historical_stats['wins'] / fact_team_historical_stats['matches_played']
)
fact_team_historical_stats['goal_difference'] = (
    fact_team_historical_stats['goals_scored'] - fact_team_historical_stats['goals_conceded']
)
fact_team_historical_stats['avg_goals_scored'] = (
    fact_team_historical_stats['goals_scored'] / fact_team_historical_stats['matches_played']
)
fact_team_historical_stats['avg_goals_conceded'] = (
    fact_team_historical_stats['goals_conceded'] / fact_team_historical_stats['matches_played']
)
fact_team_historical_stats['has_knockout_experience'] = (
    fact_team_historical_stats['knockout_matches'] > 0
).astype(int)
fact_team_historical_stats['knockout_win_rate'] = (
    fact_team_historical_stats['knockout_wins'] /
    fact_team_historical_stats['knockout_matches'].replace(0, np.nan)
).fillna(0)
# Peso 2 para cartão vermelho é arbitrário; no futebol real implica suspensão.
# Objetivo aqui é apenas ranquear indisciplina relativa.
fact_team_historical_stats['discipline_score'] = (
    fact_team_historical_stats['total_yellow_cards'] +
    2 * fact_team_historical_stats['total_red_cards']
)

titles = worldcup['Champion'].value_counts()
fact_team_historical_stats['titles'] = (
    fact_team_historical_stats['Team'].map(titles).fillna(0).astype(int)
)
fact_team_historical_stats['is_champion'] = (
    fact_team_historical_stats['titles'] > 0
).astype(int)

print(f'Teams: {len(fact_team_historical_stats)}')
fact_team_historical_stats.sort_values('wins', ascending=False).head(10)

## 5. `fact_team_year_stage_stats`

Granularity: **1 row = 1 team × year × stage**

In [ ]:
fact_team_year_stage_stats = (
    team_matches.groupby(['Team', 'Year', 'Stage', 'Era']).agg(
        matches_played=('Is_Win', 'count'),
        wins=('Is_Win', 'sum'),
        draws=('Is_Draw', 'sum'),
        losses=('Is_Loss', 'sum'),
        goals_scored=('Goals_For', 'sum'),
        goals_conceded=('Goals_Against', 'sum'),
    )
    .reset_index()
)

fact_team_year_stage_stats['win_rate'] = (
    fact_team_year_stage_stats['wins'] / fact_team_year_stage_stats['matches_played']
)
fact_team_year_stage_stats['goal_difference'] = (
    fact_team_year_stage_stats['goals_scored'] - fact_team_year_stage_stats['goals_conceded']
)
fact_team_year_stage_stats['avg_goals_scored'] = (
    fact_team_year_stage_stats['goals_scored'] / fact_team_year_stage_stats['matches_played']
)
fact_team_year_stage_stats['avg_goals_conceded'] = (
    fact_team_year_stage_stats['goals_conceded'] / fact_team_year_stage_stats['matches_played']
)

print(f'Rows: {len(fact_team_year_stage_stats)}')
fact_team_year_stage_stats[
    fact_team_year_stage_stats['Team'] == 'Brazil'
].head(10)

## 6. `fact_modern_team_stats`

Granularity: **1 row = 1 team, 2002+ (Contemporary Era)**

In [ ]:
modern_matches = team_matches[team_matches['Year'] >= 2002]

fact_modern_team_stats = (
    modern_matches.groupby('Team').agg(
        matches_played=('Is_Win', 'count'),
        wins=('Is_Win', 'sum'),
        draws=('Is_Draw', 'sum'),
        losses=('Is_Loss', 'sum'),
        goals_scored=('Goals_For', 'sum'),
        goals_conceded=('Goals_Against', 'sum'),
        knockout_matches=('Is_Knockout', 'sum'),
        knockout_wins=('Knockout_Win', 'sum'),
    )
    .reset_index()
)

fact_modern_team_stats['win_rate'] = (
    fact_modern_team_stats['wins'] / fact_modern_team_stats['matches_played']
)
fact_modern_team_stats['goal_difference'] = (
    fact_modern_team_stats['goals_scored'] - fact_modern_team_stats['goals_conceded']
)
fact_modern_team_stats['avg_goals_scored'] = (
    fact_modern_team_stats['goals_scored'] / fact_modern_team_stats['matches_played']
)
fact_modern_team_stats['avg_goals_conceded'] = (
    fact_modern_team_stats['goals_conceded'] / fact_modern_team_stats['matches_played']
)
fact_modern_team_stats['has_knockout_experience'] = (
    fact_modern_team_stats['knockout_matches'] > 0
).astype(int)
fact_modern_team_stats['knockout_win_rate'] = (
    fact_modern_team_stats['knockout_wins'] /
    fact_modern_team_stats['knockout_matches'].replace(0, np.nan)
).fillna(0)

# Filter first so MinMaxScaler normalizes within the relevant population
fact_modern_team_stats = fact_modern_team_stats[
    fact_modern_team_stats['matches_played'] >= MIN_MODERN_MATCHES
].reset_index(drop=True)

# modern_score: scale each feature to [0,1] so weights 50/30/20 reflect true proportions
_score_features = ['avg_goals_scored', 'goal_difference', 'matches_played']
_mms = MinMaxScaler()
_scaled = _mms.fit_transform(fact_modern_team_stats[_score_features])
fact_modern_team_stats['modern_score'] = (
    _scaled[:, 0] * 0.5 + _scaled[:, 1] * 0.3 + _scaled[:, 2] * 0.2
)

print(f'Teams with >={MIN_MODERN_MATCHES} modern matches: {len(fact_modern_team_stats)}')
fact_modern_team_stats.sort_values('modern_score', ascending=False).head(15)

In [ ]:
assert fact_modern_team_stats['modern_score'].between(0, 1).all(), \
    "modern_score out of [0,1] — MinMaxScaler should guarantee this"
print("modern_score scale check OK — all values in [0, 1]")

## 7. `fact_team_ml_features`

Granularity: **1 row = 1 team** — clustering + champion probability

In [ ]:
ML_FEATURES = [
    'win_rate',
    'goal_difference',
    'avg_goals_scored',
    'avg_goals_conceded',
    'knockout_win_rate',
]

X = fact_team_historical_stats[ML_FEATURES].copy()
y = fact_team_historical_stats['is_champion']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)


def label_clusters(kmeans_model, feat_scaler, feature_names):
    """Derive cluster labels from centroid characteristics to avoid unstable hardcoded IDs."""
    centroids_orig = feat_scaler.inverse_transform(kmeans_model.cluster_centers_)
    centroids_df = pd.DataFrame(centroids_orig, columns=feature_names)

    win_idx = feature_names.index('win_rate')
    ko_idx = feature_names.index('knockout_win_rate')
    # Both win_rate and knockout_win_rate are in [0,1] in original scale
    composite = centroids_orig[:, win_idx] + centroids_orig[:, ko_idx]

    rank = np.argsort(composite)  # worst → best
    ordered_labels = [
        'Emerging Teams',
        'Competitive Nations',
        'Defensive / Volatile',
        'Elite Champions',
    ]
    label_map = {int(cid): lbl for cid, lbl in zip(rank, ordered_labels)}

    print("Derived CLUSTER_LABELS:", label_map)
    print("\nCentroid table (original scale):")
    centroids_df.index.name = 'cluster_id'
    centroids_df['label'] = centroids_df.index.map(label_map)
    centroids_df['composite_win_ko'] = composite
    print(centroids_df.sort_values('composite_win_ko').round(3).to_string())

    return label_map


CLUSTER_LABELS = label_clusters(kmeans, scaler, ML_FEATURES)

# PCA for visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

var_ratio = pca.explained_variance_ratio_
print(f"\nPCA explained variance: PC1={var_ratio[0]:.1%}, PC2={var_ratio[1]:.1%}, total={sum(var_ratio[:2]):.1%}")
if sum(var_ratio[:2]) < 0.60:
    print("WARNING: total explained variance < 60% — scatter plot should be interpreted with caution.")

# Champion classifier — 'titles' excluded to prevent data leakage (target is derived from titles)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nChampions — train: {y_train.sum()} | test: {y_test.sum()}")
model = RandomForestClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
assert 'titles' not in ML_FEATURES, "Data leakage: 'titles' must not be in ML_FEATURES"
print("leakage check OK")

assert isinstance(CLUSTER_LABELS, dict) and len(CLUSTER_LABELS) == 4, \
    "CLUSTER_LABELS must be a derived dict with 4 entries"
assert set(CLUSTER_LABELS.values()) == {
    'Emerging Teams', 'Competitive Nations', 'Defensive / Volatile', 'Elite Champions'
}, "CLUSTER_LABELS values mismatch"
print("cluster labels derivation check OK")

In [ ]:
fact_team_ml_features = fact_team_historical_stats[['Team', 'titles', 'is_champion']].copy()
fact_team_ml_features['cluster'] = clusters
fact_team_ml_features['team_profile'] = fact_team_ml_features['cluster'].map(CLUSTER_LABELS)
fact_team_ml_features['pca1'] = pca_result[:, 0]
fact_team_ml_features['pca2'] = pca_result[:, 1]
fact_team_ml_features['champion_probability'] = model.predict_proba(X)[:, 1]

fact_team_ml_features.sort_values('champion_probability', ascending=False)[
    ['Team', 'champion_probability', 'titles', 'team_profile']
].head(15)

## 8. Visualizations

In [ ]:
# Cluster scatter plot
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=fact_team_ml_features,
    x='pca1', y='pca2',
    hue='team_profile',
    size='titles',
    sizes=(50, 400),
)
plt.title('World Cup Team Clusters (PCA)')
plt.show()

In [ ]:
# Feature correlation matrix
corr = fact_team_historical_stats[ML_FEATURES + ['is_champion']].corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# PCA feature loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=ML_FEATURES,
    columns=['PCA1', 'PCA2']
).round(3)
print(loadings.sort_values('PCA1', ascending=False))

## 9. Export

In [ ]:
import os
os.makedirs('../data/analytics/notebooks', exist_ok=True)

fact_team_historical_stats.to_csv(
    '../data/analytics/notebooks/fact_team_historical_stats.csv', index=False
)
fact_team_year_stage_stats.to_csv(
    '../data/analytics/notebooks/fact_team_year_stage_stats.csv', index=False
)
fact_modern_team_stats.to_csv(
    '../data/analytics/notebooks/fact_modern_team_stats.csv', index=False
)
fact_team_ml_features.to_csv(
    '../data/analytics/notebooks/fact_team_ml_features.csv', index=False
)

print('Exported:')
print(f'  fact_team_historical_stats  → {len(fact_team_historical_stats)} rows')
print(f'  fact_team_year_stage_stats  → {len(fact_team_year_stage_stats)} rows')
print(f'  fact_modern_team_stats      → {len(fact_modern_team_stats)} rows')
print(f'  fact_team_ml_features       → {len(fact_team_ml_features)} rows')

## Changelog

- **CRÍTICO 1** — Removido `'titles'` de `ML_FEATURES` (data leakage: target `is_champion` é derivado de `titles`); adicionado `stratify=y` no `train_test_split` e `class_weight='balanced'` no `RandomForestClassifier`.
- **CRÍTICO 2** — `modern_score` agora aplica `MinMaxScaler` nas três features antes da combinação ponderada, garantindo que os pesos 50/30/20 sejam respeitados em vez de serem dominados pela escala de `matches_played`.
- **CRÍTICO 3** — `CLUSTER_LABELS` derivado dinamicamente via `label_clusters()` a partir dos centroides invertidos (composite = `win_rate + knockout_win_rate`), eliminando dependência de IDs instáveis do KMeans.
- **IMPORTANTE 1** — Adicionada coluna `has_knockout_experience` em `fact_team_historical_stats` e `fact_modern_team_stats` para distinguir times sem mata-mata de times que perderam todos.
- **IMPORTANTE 2** — Adicionado comentário explicando as escolhas do `TEAM_NAME_MAP`; coluna `Team_Original` em `team_matches` preserva nomes antes do remap.
- **IMPORTANTE 3** — Substituídas lambdas em todos os `groupby` por colunas indicadoras vetorizadas (`Is_Win`, `Is_Draw`, `Is_Loss`); `matches_played` usa `('Is_Win', 'count')` para deixar a intenção clara.
- **IMPORTANTE 4** — PCA explained variance exibida com alerta se < 60%; `MIN_MODERN_MATCHES = 10` substitui número mágico; comentário sobre arbitrariedade do peso 2 no `discipline_score`.
- **PEQUENO** — Paths definidos com `pathlib.Path` e fallback para ambientes sem `__file__`; comentário explicando intervalos resultantes do `pd.cut` de era.